In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [5]:
# URL yang akan di scraping
URL = "https://liquipedia.net/mobilelegends/Portal:Tournaments"

# Header untuk menghindari blocking 403
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Linux; Cashtree) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/34.0.1847.114 Mobile Safari/537.36"
}

In [6]:
response = requests.get(URL, headers=HEADERS, timeout=15)

In [7]:
if response.status_code != 200:
    print(f"Error: HTTP {response.status_code}")

soup = BeautifulSoup(response.content, "html.parser")

## Get Tournament List

In [37]:
main_content = soup.find("div", {"class": "mw-content-ltr"})

In [38]:
tournament_tab_header = main_content.find("div", {"class": "tabs-static"})

In [39]:
headers_menu = tournament_tab_header.find_all("li")

In [40]:
base_img_url = "https://liquipedia.net"

In [194]:
tournaments = []

In [195]:
for h in headers_menu:
    tab_name = h.text
    if tab_name  not in skip_menu:
        header_name = tab_name.replace("\xa0", " ")
        link = f"{base_img_url}{h.find("a")["href"]}"
        data_tournament = get_tournament_data(link, header_name)
        tournaments.append(data_tournament)
        

finish scraping data Recent Results...
finish scraping data S-Tier Events...
finish scraping data A-Tier Events...
finish scraping data B-Tier Events...
finish scraping data C-Tier Events...
finish scraping data D-Tier Events...
finish scraping data Monthly...
finish scraping data Weekly...
finish scraping data Qualifiers...
finish scraping data Misc...
finish scraping data Show Matches...
finish scraping data National...


In [ ]:
tournaments

In [25]:
skip_menu = ["Introduction", "Recent Results"]

In [42]:
tournament_tier = main_content.find_all("h2")

In [44]:
for t in tournament_tier:
    print(t.text)

About S-Tier, A-Tier, B-Tier, C-Tier[edit]
Contents
Upcoming and Most Recent Tournaments[edit]


In [45]:
tournament_url = "https://liquipedia.net/mobilelegends/Recent_Tournament_Results"

In [173]:
base_url = "https://liquipedia.net"

In [193]:
def get_tournament_data(url, name):
    data = {"tournament": name, "list": []}
    tournament_request = requests.get(tournament_url, headers=HEADERS, timeout=15)
    if tournament_request.status_code != 200:
        print(f"Error: HTTP {tournament_request.status_code}")

    soup = BeautifulSoup(tournament_request.content, "html.parser")
    main_content = soup.find("div", {"class": "mw-content-ltr"})
    tournamen_name = main_content.find_all("h2")

    skip_content = ["Contents"]
    for tier in tournamen_name:
        if tier.text not in skip_content:
            list_tournament = tier.find_next_sibling().find_all("div", {"class": "gridRow"})
            if list_tournament is not None:
                for tour in list_tournament:
                    # print(tour.select_one("div.gridCell.Tournament.Header a")["title"])
                    winner = tour.select_one("div.gridCell.FirstPlace span.Participants span.name a")
                    runner_up = tour.select_one("div.gridCell.SecondPlace span.Participants span.name a")
                    partisipants = tour.select_one("div.gridCell.PlayerNumber")
                    link = tour.select_one("div.gridCell.Tournament.Header span.league-icon-small-image.darkmode")
                    if link is None:
                        link = tour.select_one("div.gridCell.Tournament.Header span.league-icon-small-image")

                    link = link.find_next_sibling()["href"]
                    tournament = {
                        "name": tour.select_one("div.gridCell.Tournament.Header a")["title"],
                        "link": f"{base_url}{link}",
                        "date": tour.select_one("div.gridCell.Date").text,
                        "location": tour.select_one("div.gridCell.Location").text.strip(),
                        "prize_pool": tour.select_one("div.gridCell.Prize").text.strip(),
                        "partisipants": partisipants.text.replace(partisipants.find("span", {"class": "PlayerNumberSuffix"}).text if partisipants.find("span", {"class": "PlayerNumberSuffix"}) else "", ""),
                        "winner": winner.text if winner else "TBD",
                        "runner_up": runner_up.text if runner_up else "TBD"
                    }
    
                    data["list"].append(tournament)


    print(f"finish scraping data {name}...")
    return data
                    

In [ ]:
tournamen = get_tournament_data(tournament_url, "Recent Results")

## Get Match Per Tournament

In [327]:
def get_match_per_tournament(tournamen_url, name):
    tournament_request = requests.get(tournamen_url, headers=HEADERS, timeout=15)
    if tournament_request.status_code != 200:
        print(f"Error: HTTP {tournament_request.status_code}")

    soup = BeautifulSoup(tournament_request.content, "html.parser")
    main_content = soup.find("div", {"class": "mw-content-ltr"})
    headers_content = main_content.select_one("div.tabs-static ul.navigation-not-searchable")
    tabs_header = headers_content.find_all("li")
    tournament_data = {"name": name, "link": tournamen_url, "partisipants": []}
    for t in tabs_header:
        if t.text == "Overview":
            partisipants = main_content.find_all("div", {"class": "teamcard-columns-4"})
            for p in partisipants:
                team_with_players = p.find_all("div", {"class": "template-box"})
                for tm in team_with_players:
                    partisipant = {"players": [], "staff": []}
                    if tm.select_one("div.teamcard center") is not None:
                        team_part = tm.select_one("div.teamcard center span.flag").find_next_sibling()
                        partisipant["team"] = team_part.text
                        team_status_content = tm.select_one("div.teamcard center").find_next_sibling()
                        qualifier = team_status_content.select_one("table.wikitable.wikitable-bordered.logo tbody tr td.teamcard-qualifier")
                        if qualifier is not None:
                            partisipant["qualifier"] = qualifier.text

                        # players
                        roster_content = team_status_content.select('table.wikitable.wikitable-bordered.list[data-toggle-area-content="1"] tbody tr')
                        if len(roster_content) == 0:
                            roster_content = team_status_content.find_next_sibling().select('table.wikitable.wikitable-bordered.list[data-toggle-area-content="1"] tbody tr')

                        player = {}
                        for rs in roster_content:
                            
                            role = rs.select_one("th img")
                            if role is not None:
                                player_name = rs.select_one("td span.flag").find_next_sibling()
                                player["_".join(role["alt"].split(" "))] = player_name.text
                        partisipant["players"] = player


                        # staff
                        staff_content = team_status_content.select('table.wikitable.wikitable-bordered.list[data-toggle-area-content="3"] tbody tr')
                        if len(staff_content) == 0:
                            staff_content = team_status_content.find_next_sibling().select('table.wikitable.wikitable-bordered.list[data-toggle-area-content="3"] tbody tr')
                            # print(f"Team {team_part.text} staff: {len(staff_content)}")
                        for st in staff_content:
                            staff = {}
                            role = st.select_one("th")
                            if role is not None:
                                role_name = role.find("img")
                                if role_name is None:
                                    role_name = st.select_one("th span abbr")
                                    role_name = role_name["title"]
                                else:
                                    role_name = role_name["alt"]
    
                                coach_name = st.select_one("td span.flag").find_next_sibling()
                                staff["_".join(role_name.split(" "))] = coach_name.text
                                partisipant["staff"].append(staff)
                        # print(partisipant)
                        tournament_data["partisipants"].append(partisipant)
    print(tournament_data)
                

In [ ]:
tournament_per_match = get_match_per_tournament("https://liquipedia.net/mobilelegends/MSC/2025", "MLBB Mid Season Cup 2025")